# Download Aerial Photos of Quarries

## Notes
- I had to modify one `Abrasion Resistance` from `"3,234"` to `3234`

In [2]:
import os
import polars as pl
import requests

from pyproj import Transformer

In [3]:
with open(".google_api_key.txt", "r") as f:
    API_KEY = f.read()

In [4]:
# Create a transformer: OSGB36 → WGS84
transformer = Transformer.from_crs(
    "EPSG:27700",  # British National Grid (Easting/Northing)
    "EPSG:4326",   # WGS84 (Latitude/Longitude)
    always_xy=True # Ensure input as (Easting, Northing) → output (Longitude, Latitude)
)

df = pl.read_csv("quarries.csv")

rows = []

for idx, row in enumerate(df.iter_rows(named=True), start=1):
    easting = row["X (easting)"]
    northing = row["Y (northing)"]

    if easting is None or northing is None:
        continue

    lon, lat = transformer.transform(easting, northing)

    rows.append((idx, (lat, lon)))

In [5]:
def download_satellite_image(index, lat, lon, api_key, zoom=18, size="640x640"):
    url = (
        "https://maps.googleapis.com/maps/api/staticmap"
        f"?center={lat},{lon}"
        f"&zoom={zoom}"
        f"&size={size}"
        f"&maptype=satellite"
        f"&key={api_key}"
    )

    response = requests.get(url)
    if response.status_code == 200:
        filename = os.path.join("images", f"{index}.png")
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"❌ Failed to fetch image: {response.status_code} - {response.text}")

In [10]:
for index, (lat, long) in rows:
    download_satellite_image(index, lat, long, API_KEY, zoom=16)